# tsfresh Feature Extraction + Filtering
Extract time series features using tsfresh, then apply 3 filtering gates:
1. **Filter 1**: Statistical relevance (p-value < 0.05)
2. **Filter 2**: Redundancy pruning (|corr| > 0.9 → keep one)
3. **Filter 3**: Temporal stability (stable across time folds)

In [1]:
import polars as pl
import pandas as pd
import numpy as np
import json
from pathlib import Path
from tsfresh import extract_features, select_features
from tsfresh.feature_extraction import MinimalFCParameters, EfficientFCParameters
from tsfresh.utilities.dataframe_functions import impute
from scipy import stats
from sklearn.feature_selection import mutual_info_regression
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Paths
DATA_DIR = Path("../../data")
RAW_DIR = DATA_DIR / "raw/combined"
PROCESSED_DIR = DATA_DIR / "processed"
FILTER_RESULTS = PROCESSED_DIR / "tsfresh_feature_filter_results.json"
FILTERED_DATA = PROCESSED_DIR / "tsfresh_filtered_data.parquet"
OUTPUT_DIR = PROCESSED_DIR / "tsfresh_features"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

In [3]:
# Filter configuration
FILTER_CONFIG = {
    # Filter 1: Statistical relevance
    "alpha": 0.05,            # p-value threshold
    
    # Filter 2: Redundancy pruning
    "corr_threshold": 0.90,   # |corr| > 0.9 → drop duplicates
    
    # Filter 3: Temporal stability
    "n_time_folds": 3,        # number of time-based folds
    "stability_threshold": 0.5,  # feature must be relevant in >50% of folds
    "importance_cv_max": 1.0,    # max CV of importance across folds
}

## Load Data

In [4]:
# Load filter results
with open(FILTER_RESULTS, "r") as f:
    filter_results = json.load(f)

FEATURES = filter_results["tsfresh_features_final"]
print(f"Features to process: {len(FEATURES)}")

Features to process: 68


In [5]:
# Load filtered data + target
df_filtered = pl.read_parquet(FILTERED_DATA)
df_raw = pl.read_parquet(RAW_DIR / "train.parquet")

# Join target column
df = df_filtered.join(
    df_raw.select(["sub_code", "ts_index", "y_target"]),
    on=["sub_code", "ts_index"],
    how="left"
)

print(f"Data shape: {df.shape}")
print(f"Unique entities: {df.select('sub_code').n_unique()}")

: 

: 

: 

## tsfresh Configuration

In [3]:
# tsfresh settings
TSFRESH_CONFIG = {
    "parameters": "minimal",
    "n_jobs": 4,
    "chunksize": 10,
    "disable_progress": False,
}

# Sampling (set to None for full data)
SAMPLE_ENTITIES = None  # All entities
SAMPLE_FEATURES = 10    # First 10 features for testing

if TSFRESH_CONFIG["parameters"] == "minimal":
    extraction_settings = MinimalFCParameters()
else:
    extraction_settings = EfficientFCParameters()

print(f"Using: {TSFRESH_CONFIG['parameters']} parameters")

NameError: name 'MinimalFCParameters' is not defined

In [4]:
# Sample if needed
if SAMPLE_ENTITIES:
    unique_entities = df.select("sub_code").unique()
    sample_entities = unique_entities.sample(n=min(SAMPLE_ENTITIES, len(unique_entities)))
    df_sample = df.filter(pl.col("sub_code").is_in(sample_entities.to_series()))
else:
    df_sample = df

features_to_use = FEATURES[:SAMPLE_FEATURES] if SAMPLE_FEATURES else FEATURES

print(f"Entities: {df_sample.select('sub_code').n_unique()}")
print(f"Features: {len(features_to_use)}")

NameError: name 'df' is not defined

## Prepare Data (Handle NaN)

In [5]:
# Select columns
cols_for_tsfresh = ["sub_code", "ts_index"] + features_to_use
df_tsfresh = df_sample.select(cols_for_tsfresh).to_pandas()

# Handle NaN: forward/backward fill per entity
df_tsfresh = df_tsfresh.sort_values(["sub_code", "ts_index"])
for col in features_to_use:
    df_tsfresh[col] = df_tsfresh.groupby("sub_code")[col].transform(
        lambda x: x.ffill().bfill()
    )
df_tsfresh[features_to_use] = df_tsfresh[features_to_use].fillna(0)

# Ensure types
df_tsfresh["sub_code"] = df_tsfresh["sub_code"].astype(str)
df_tsfresh["ts_index"] = df_tsfresh["ts_index"].astype(int)

print(f"DataFrame ready: {df_tsfresh.shape}")

NameError: name 'features_to_use' is not defined

## Extract Features with tsfresh

In [6]:
%%time

extracted_features = extract_features(
    df_tsfresh,
    column_id="sub_code",
    column_sort="ts_index",
    default_fc_parameters=extraction_settings,
    n_jobs=TSFRESH_CONFIG["n_jobs"],
    chunksize=TSFRESH_CONFIG["chunksize"],
    disable_progressbar=TSFRESH_CONFIG["disable_progress"],
)

# Impute any NaN in output
extracted_features = impute(extracted_features)

print(f"\nExtracted: {extracted_features.shape}")

CPU times: user 6 μs, sys: 1 μs, total: 7 μs
Wall time: 14.1 μs


NameError: name 'extract_features' is not defined

In [7]:
# Create target series aligned with extracted features
target_df = (
    df_sample.group_by("sub_code")
    .agg(pl.col("y_target").mean().alias("y_target"))
    .to_pandas()
    .set_index("sub_code")
)

# Align indices
common_idx = extracted_features.index.intersection(target_df.index)
X = extracted_features.loc[common_idx]
y = target_df.loc[common_idx, "y_target"]

# Drop NaN targets
valid_mask = ~y.isna()
X = X.loc[valid_mask]
y = y.loc[valid_mask]

print(f"Aligned data: X={X.shape}, y={len(y)}")

NameError: name 'df_sample' is not defined

---
# 🧱 FILTER 1: Statistical Relevance
Keep features with p-value < α (univariate hypothesis testing)

In [8]:
def compute_relevance(X, y, alpha=0.05):
    """Compute p-values for each feature using Spearman correlation"""
    results = {}
    
    for col in X.columns:
        x = X[col].values
        
        # Skip constant features
        if np.std(x) < 1e-10:
            results[col] = {"p_value": 1.0, "relevant": False, "correlation": 0.0}
            continue
        
        # Spearman correlation test
        corr, p_value = stats.spearmanr(x, y)
        
        results[col] = {
            "p_value": p_value if not np.isnan(p_value) else 1.0,
            "correlation": corr if not np.isnan(corr) else 0.0,
            "relevant": p_value < alpha if not np.isnan(p_value) else False
        }
    
    return results

# Compute relevance
relevance_results = compute_relevance(X, y, FILTER_CONFIG["alpha"])

# Filter relevant features
relevant_features = [f for f, r in relevance_results.items() if r["relevant"]]

print(f"🧱 FILTER 1 (Statistical Relevance)")
print(f"   Total features: {len(X.columns)}")
print(f"   Relevant (p < {FILTER_CONFIG['alpha']}): {len(relevant_features)}")
print(f"   Dropped: {len(X.columns) - len(relevant_features)}")

NameError: name 'X' is not defined

In [9]:
# Show top relevant features by p-value
sorted_by_pval = sorted(relevance_results.items(), key=lambda x: x[1]["p_value"])
print("\nTop 10 most relevant features:")
print(f"{'Feature':<50} {'p-value':>12} {'Corr':>10}")
print("-" * 75)
for feat, res in sorted_by_pval[:10]:
    print(f"{feat:<50} {res['p_value']:>12.2e} {res['correlation']:>10.4f}")

NameError: name 'relevance_results' is not defined

In [10]:
# Apply Filter 1
X_filter1 = X[relevant_features]
print(f"\nAfter Filter 1: {X_filter1.shape}")

NameError: name 'X' is not defined

---
# 🧱 FILTER 2: Redundancy Pruning
If |corr| > 0.9 → keep simpler feature, drop rest

In [11]:
def get_feature_complexity(feature_name):
    """Score feature complexity (lower = simpler, prefer to keep)"""
    # Simple stats are preferred
    simple_stats = ["mean", "median", "sum", "min", "max", "std", "length"]
    medium_stats = ["variance", "abs", "quantile", "root"]
    
    name_lower = feature_name.lower()
    
    for stat in simple_stats:
        if stat in name_lower:
            return 1
    
    for stat in medium_stats:
        if stat in name_lower:
            return 2
    
    return 3  # Complex


def prune_redundant_features(X, threshold=0.9):
    """Remove highly correlated features, keeping simpler ones"""
    
    # Compute correlation matrix
    corr_matrix = X.corr().abs()
    
    # Track which features to drop
    features_to_drop = set()
    features = list(X.columns)
    
    for i, feat_i in enumerate(features):
        if feat_i in features_to_drop:
            continue
            
        for feat_j in features[i+1:]:
            if feat_j in features_to_drop:
                continue
            
            corr = corr_matrix.loc[feat_i, feat_j]
            
            if corr > threshold:
                # Keep simpler feature
                comp_i = get_feature_complexity(feat_i)
                comp_j = get_feature_complexity(feat_j)
                
                if comp_i <= comp_j:
                    features_to_drop.add(feat_j)
                else:
                    features_to_drop.add(feat_i)
    
    kept_features = [f for f in features if f not in features_to_drop]
    return kept_features, features_to_drop

# Apply redundancy pruning
kept_features, dropped_redundant = prune_redundant_features(
    X_filter1, 
    FILTER_CONFIG["corr_threshold"]
)

print(f"🧱 FILTER 2 (Redundancy Pruning)")
print(f"   Input features: {len(X_filter1.columns)}")
print(f"   Kept (unique): {len(kept_features)}")
print(f"   Dropped (|corr| > {FILTER_CONFIG['corr_threshold']}): {len(dropped_redundant)}")

NameError: name 'X_filter1' is not defined

In [12]:
# Apply Filter 2
X_filter2 = X_filter1[kept_features]
print(f"\nAfter Filter 2: {X_filter2.shape}")

NameError: name 'X_filter1' is not defined

---
# 🧱 FILTER 3: Temporal Stability
Feature must be relevant across multiple time folds (not just one period)

In [13]:
def compute_temporal_stability(df_full, features, target_col, n_folds=3, alpha=0.05):
    """Check feature relevance across time-based folds"""
    
    # Get time range
    df_pd = df_full.select(["sub_code", "ts_index", target_col] + features).to_pandas()
    
    # Sort by time and split into folds
    df_pd = df_pd.sort_values("ts_index")
    time_values = df_pd["ts_index"].unique()
    fold_size = len(time_values) // n_folds
    
    fold_results = {feat: [] for feat in features}
    
    for fold_idx in range(n_folds):
        # Get fold time range
        start_idx = fold_idx * fold_size
        end_idx = start_idx + fold_size if fold_idx < n_folds - 1 else len(time_values)
        fold_times = time_values[start_idx:end_idx]
        
        # Filter data for this fold
        fold_data = df_pd[df_pd["ts_index"].isin(fold_times)]
        
        # Aggregate per entity
        fold_agg = fold_data.groupby("sub_code").agg({
            **{feat: "mean" for feat in features},
            target_col: "mean"
        }).dropna()
        
        if len(fold_agg) < 10:
            continue
        
        y_fold = fold_agg[target_col]
        
        # Test relevance for each feature in this fold
        for feat in features:
            x = fold_agg[feat].values
            if np.std(x) < 1e-10:
                fold_results[feat].append({"relevant": False, "importance": 0.0})
                continue
            
            corr, p_value = stats.spearmanr(x, y_fold)
            is_relevant = p_value < alpha if not np.isnan(p_value) else False
            importance = abs(corr) if not np.isnan(corr) else 0.0
            
            fold_results[feat].append({
                "relevant": is_relevant,
                "importance": importance,
                "correlation": corr if not np.isnan(corr) else 0.0
            })
    
    # Compute stability metrics
    stability_metrics = {}
    for feat, folds in fold_results.items():
        if not folds:
            stability_metrics[feat] = {"relevance_ratio": 0, "importance_cv": np.inf, "stable": False}
            continue
        
        n_relevant = sum(1 for f in folds if f["relevant"])
        relevance_ratio = n_relevant / len(folds)
        
        importances = [f["importance"] for f in folds]
        mean_imp = np.mean(importances)
        std_imp = np.std(importances)
        cv = std_imp / mean_imp if mean_imp > 1e-10 else np.inf
        
        # Check sign consistency
        correlations = [f.get("correlation", 0) for f in folds]
        signs = [np.sign(c) for c in correlations if c != 0]
        sign_consistent = len(set(signs)) <= 1 if signs else False
        
        stability_metrics[feat] = {
            "relevance_ratio": relevance_ratio,
            "importance_cv": cv,
            "mean_importance": mean_imp,
            "sign_consistent": sign_consistent,
            "fold_details": folds,
        }
    
    return stability_metrics

In [14]:
# Get original feature names (before tsfresh transformation)
# Map tsfresh features back to their source
tsfresh_to_source = {}
for col in X_filter2.columns:
    source = col.split("__")[0]
    tsfresh_to_source[col] = source

# Get unique source features
source_features = list(set(tsfresh_to_source.values()))
source_features = [f for f in source_features if f in df_sample.columns]

print(f"Source features to test stability: {len(source_features)}")

NameError: name 'X_filter2' is not defined

In [15]:
# Compute temporal stability on SOURCE features
stability_metrics = compute_temporal_stability(
    df_sample,
    source_features,
    "y_target",
    n_folds=FILTER_CONFIG["n_time_folds"],
    alpha=FILTER_CONFIG["alpha"]
)

# Identify stable source features
stable_sources = set()
for feat, metrics in stability_metrics.items():
    is_stable = (
        metrics["relevance_ratio"] >= FILTER_CONFIG["stability_threshold"] and
        metrics["importance_cv"] <= FILTER_CONFIG["importance_cv_max"]
    )
    if is_stable:
        stable_sources.add(feat)

print(f"🧱 FILTER 3 (Temporal Stability)")
print(f"   Source features tested: {len(source_features)}")
print(f"   Stable sources: {len(stable_sources)}")
print(f"   Unstable sources: {len(source_features) - len(stable_sources)}")

NameError: name 'df_sample' is not defined

In [ ]:
# Show stability metrics
print("\nStability metrics per source feature:")
print(f"{'Feature':<20} {'Rel.Ratio':>10} {'Imp.CV':>10} {'MeanImp':>10} {'Stable':>8}")
print("-" * 65)
for feat, m in sorted(stability_metrics.items(), key=lambda x: -x[1]["relevance_ratio"]):
    is_stable = feat in stable_sources
    print(f"{feat:<20} {m['relevance_ratio']:>10.2f} {m['importance_cv']:>10.2f} {m['mean_importance']:>10.4f} {'✓' if is_stable else '✗':>8}")

In [ ]:
# Filter tsfresh features based on stable sources
stable_tsfresh_features = [
    col for col in X_filter2.columns 
    if tsfresh_to_source.get(col) in stable_sources
]

X_filter3 = X_filter2[stable_tsfresh_features]
print(f"\nAfter Filter 3: {X_filter3.shape}")

---
# Final Summary

In [ ]:
print("=" * 60)
print("FEATURE FILTERING SUMMARY")
print("=" * 60)
print(f"\n📊 Starting features: {len(extracted_features.columns)}")
print(f"\n🧱 Filter 1 (Relevance, p<{FILTER_CONFIG['alpha']}):")
print(f"   Kept: {len(X_filter1.columns)} | Dropped: {len(extracted_features.columns) - len(X_filter1.columns)}")
print(f"\n🧱 Filter 2 (Redundancy, |corr|>{FILTER_CONFIG['corr_threshold']}):")
print(f"   Kept: {len(X_filter2.columns)} | Dropped: {len(X_filter1.columns) - len(X_filter2.columns)}")
print(f"\n🧱 Filter 3 (Temporal Stability):")
print(f"   Kept: {len(X_filter3.columns)} | Dropped: {len(X_filter2.columns) - len(X_filter3.columns)}")
print(f"\n✅ FINAL: {len(X_filter3.columns)} features")

## Save Final Features

In [ ]:
# Save filtered features
output_df = X_filter3.reset_index()
output_df = output_df.rename(columns={"index": "sub_code"})
output_df.to_parquet(OUTPUT_DIR / "tsfresh_filtered_final.parquet", index=False)

print(f"✅ Final features saved: {OUTPUT_DIR / 'tsfresh_filtered_final.parquet'}")
print(f"   Shape: {output_df.shape}")

In [ ]:
# Save filter results
filter_output = {
    "config": FILTER_CONFIG,
    "extraction_params": TSFRESH_CONFIG["parameters"],
    "counts": {
        "extracted": len(extracted_features.columns),
        "after_relevance": len(X_filter1.columns),
        "after_redundancy": len(X_filter2.columns),
        "after_stability": len(X_filter3.columns),
    },
    "final_features": list(X_filter3.columns),
    "stable_source_features": list(stable_sources),
    "relevance_results": {k: {"p_value": v["p_value"], "relevant": v["relevant"]} 
                          for k, v in relevance_results.items()},
    "dropped_redundant": list(dropped_redundant),
    "stability_metrics": {k: {"relevance_ratio": v["relevance_ratio"], 
                              "importance_cv": v["importance_cv"],
                              "stable": k in stable_sources}
                          for k, v in stability_metrics.items()},
}

with open(OUTPUT_DIR / "tsfresh_filter_results.json", "w") as f:
    json.dump(filter_output, f, indent=2, default=str)

print(f"✅ Filter results saved: {OUTPUT_DIR / 'tsfresh_filter_results.json'}")